# TD 5 — Élan et surapprentissage : lire les courbes d'apprentissage

## Objectifs

À la fin de la séance, vous saurez :

- distinguer l'**erreur d'optimisation** (coût d'entraînement) de l'**écart train/test** (capacité à généraliser) ;
- reconnaître les **régimes** d'une courbe d'apprentissage : descente, plateau, remontée (surapprentissage), divergence ;
- **prédire puis mesurer** l'effet du coefficient d'élan $\mu$ sur la dynamique d'apprentissage ;
- distinguer un **zigzag systématique** (causé par l'élan) d'un **bruit de batch** (causé par les mini-lots).

## Plan de séance (≈ 3 h)

1. **Exercice 1** — un MLP sur données 2D : courbes train/test et surapprentissage.
2. **Exercice 2** — ajout de l'élan : implémentation et effet de $\mu$.
3. **Exercice 3** — lire les régimes des courbes d'apprentissage.
4. **Exercice 4** — zigzag systématique vs bruit de batch.
5. **Bilan** — synthèse et règles de lecture des courbes.

## Rappel : règle de mise à jour avec élan (momentum)

$$v_l \leftarrow \mu\, v_l + \eta\, \frac{\partial E}{\partial W_l}, \qquad W_l \leftarrow W_l - v_l$$

La **vitesse** $v_l$ accumule les gradients passés, pondérés par le facteur d'élan $\mu \in [0, 1[$ : le pas effectif est plus grand quand le gradient garde le même sens, et les oscillations sont amorties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z: Mat) -> Mat:
    """
    σ'(z) = σ(z) · (1 - σ(z)).
    """
    s = sigmoid(z)
    return s * (1.0 - s)

def tanh_prime(z: Mat) -> Mat:
    """
    tanh'(z) = 1 - tanh²(z).
    """
    return 1.0 - np.tanh(z) ** 2

ACTIVATIONS = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "tanh": (np.tanh, tanh_prime),
    "linear": (lambda z: z, lambda z: np.ones_like(z)),
}

In [ ]:
class MLP:
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.Ws = []
        self.bs = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.Ws.append(rng.normal(0.0, 0.5, size=(n_out, n_in)))
            self.bs.append(np.zeros(n_out))
        self.activation, self.activation_prime = ACTIVATIONS[activation]
        self.output_activation, self.output_activation_prime = ACTIVATIONS[output_activation]

    def predict(self, X):
        return self.forward(X)[0][-1]

    def loss(self, X: Mat, t: Mat) -> float:
        """Coût du modèle : erreur quadratique moyenne.

        E = (1/2) · mean(Σ_k (Y_k - T_k)²).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        float
            Coût quadratique moyen sur le batch.
        """
        # coût du cours : moyenne sur les échantillons de 1/2 · Σ_k (Y_k - T_k)^2
        return 0.5 * float(np.mean(np.sum((self.predict(X) - t) ** 2, axis=1)))

    def forward(self, X: Mat) -> tuple[list[Mat], list[Mat]]:
        """Passe avant du MLP, couche par couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).

        Returns
        -------
        tuple[list[Mat], list[Mat]]
            (As, Zs) : activations (avec Y0 = X en tête) et pré-activations
            de chaque couche.
        """
        A = X
        As = [A]
        Zs = []
        n = len(self.Ws)
        for l, (W, b) in enumerate(zip(self.Ws, self.bs)):
            Z = A @ W.T + b
            Zs.append(Z)
            f = self.output_activation if l == n - 1 else self.activation
            A = f(Z)
            As.append(A)
        return As, Zs

    def backward(self, X: Mat, t: Mat) -> list[tuple[Mat, Vec]]:
        """Passe arrière : gradients du coût pour chaque couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        list[tuple[Mat, Vec]]
            Liste de (dW_l, db_l), une paire de gradients par couche.
        """
        As, Zs = self.forward(X)
        N = X.shape[0]
        n = len(self.Ws)
        delta = (As[-1] - t) * self.output_activation_prime(Zs[-1])
        dWs, dbs = [], []
        for l in range(n - 1, -1, -1):
            dW = delta.T @ As[l] / N
            db = delta.sum(axis=0) / N
            dWs.insert(0, dW)
            dbs.insert(0, db)
            if l > 0:
                delta = (delta @ self.Ws[l]) * self.activation_prime(Zs[l - 1])
        return list(zip(dWs, dbs))

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Effectue une étape de descente de gradient (mise à jour des poids).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).
        lr : float
            Pas d'apprentissage.
        """
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.Ws[l] -= lr * dW
            self.bs[l] -= lr * db

    def fit_batch(self, X: Mat, t: Mat, X_test: Mat | None = None, t_test: Mat | None = None, lr: float = 0.1, epochs: int = 2000) -> dict[str, list[float]]:
        """Entraîne le MLP par descente de gradient par lot.

        Si ``X_test``/``t_test`` sont fournis, le coût de test est aussi
        ajouté à l'historique.

        Parameters
        ----------
        X : Mat
            Batch d'entraînement, de forme (N, d).
        t : Mat
            Cibles d'entraînement, de forme (N, k).
        X_test : Mat, optionnel
            Batch de test, de forme (N_test, d).
        t_test : Mat, optionnel
            Cibles de test, de forme (N_test, k).
        lr : float, optionnel
            Pas d'apprentissage (défaut : 0.1).
        epochs : int, optionnel
            Nombre maximal d'epochs (défaut : 2000).

        Returns
        -------
        dict[str, list[float]]
            Historique des coûts ``{"train": [...], "test": [...]}``.
        """
        history = {"train": []}
        if X_test is not None:
            history["test"] = []
        for epoch in range(epochs):
            self.step(X, t, lr)
            history["train"].append(self.loss(X, t))
            if X_test is not None:
                history["test"].append(self.loss(X_test, t_test))
        return history


In [ ]:
class MLPMomentum(MLP):
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0, momentum=0.9):
        super().__init__(sizes, activation, output_activation, seed)
        self.momentum = momentum
        self.vWs = [np.zeros_like(W) for W in self.Ws]
        self.vbs = [np.zeros_like(b) for b in self.bs]

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Une étape de descente de gradient avec élan (momentum).

        v_l <- mu * v_l + lr * dW_l ; W_l <- W_l - v_l.
        """
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.vWs[l] = self.momentum * self.vWs[l] + lr * dW
            self.vbs[l] = self.momentum * self.vbs[l] + lr * db
            self.Ws[l] -= self.vWs[l]
            self.bs[l] -= self.vbs[l]


In [ ]:
def gen_2d(n=500, noise=0.1, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3.0, 3.0, size=(n, 2))
    t = (np.sin(X[:, 0]) * np.cos(X[:, 1])).reshape(-1, 1) + rng.normal(0.0, noise, size=(n, 1))
    return X, t

def plot_history(history, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(history["train"], label="train")
    if history["test"]:
        ax.plot(history["test"], label="test")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE")
    ax.set_yscale("log")
    ax.legend()
    ax.grid(True)
    return ax

def split(X: Mat, t: Mat, n_train: int, seed: int = 0) -> tuple[Mat, Mat, Mat, Mat]:
    """Sépare aléatoirement un jeu de données en entraînement et test.

    Les échantillons sont mélangés par une permutation déterministe
    (graine ``seed``), puis découpés : les ``n_train`` premiers forment
    l'ensemble d'entraînement, les suivants l'ensemble de test.

    Parameters
    ----------
    X : Mat
        Données d'entrée, de forme (N, d).
    t : Mat
        Cibles, de même forme que la sortie attendue (N, k).
    n_train : int
        Taille de l'ensemble d'entraînement.
    seed : int, optionnel
        Graine du générateur aléatoire (défaut : 0).

    Returns
    -------
    tuple[Mat, Mat, Mat, Mat]
        ``(X_train, t_train, X_test, t_test)``.
    """
    rng = np.random.default_rng(seed)
    perm = rng.permutation(X.shape[0])
    X_shuf, t_shuf = X[perm], t[perm]
    return X_shuf[:n_train], t_shuf[:n_train], X_shuf[n_train:], t_shuf[n_train:]


## Exercice 1 — Courbes d'apprentissage d'un MLP

Passons à l'exercice 1.